# Name That LLM — frozen 75/25 CPU candidate

Use **CPU / Accelerator None**, **Internet Off**, and attach official competition data.
The notebook fits two classifiers once on all 2,699 official training rows and
writes `/kaggle/working/submission.parquet` with `id, model` for every actual test row.
It downloads nothing. The three-row public test is only an interface example.
Full-data fitting is for submission inference; the previously used holdout is
never evaluated. Selection is frozen: 75% original style, 25% partial centering.
The development gain over the previous best was uncertain; there is no new
leaderboard result. Saving/running a notebook is not a competition submission.

Runtime versions, two-fit cost, probabilities, output hashes, and a durable
exclusive attempt claim are recorded. An existing attempt or output is never
silently overwritten; a failed attempt requires review before any new run.
The full test batch is clustered using text alone; no test prompt IDs, known
class counts, or per-class assignment rules are used. This step uses quadratic
memory in test row count: admission requires `64 * test_rows**2 <= 1 GiB`
(4,000 rows pass). Training centering retains its fixed 700 MiB allocation
admission bound. These admission estimates are not total-process RAM limits.
Package mismatches are explicit, not hidden behind
an exact-reproduction claim.


In [ ]:
import os
os.environ.setdefault('OMP_NUM_THREADS', '2')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '2')
os.environ.setdefault('VECLIB_MAXIMUM_THREADS', '2')
import gc
import importlib.metadata
import inspect
import json
import platform
import sys
import time
import types
import warnings
from pathlib import Path
import numpy as np
from scipy import sparse
from sklearn.cluster import AgglomerativeClustering as _AgglomerativeClustering
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import normalize
from threadpoolctl import threadpool_limits

if 'NTL_INPUT_DIR' in os.environ:
    INPUT = Path(os.environ['NTL_INPUT_DIR'])
else:
    INPUT = Path('/kaggle/input/competitions/name-that-llm')
    fallback = Path('/kaggle/input/name-that-llm')
    if not INPUT.exists() and fallback.exists():
        INPUT = fallback
OUTPUT = Path(os.environ.get('NTL_OUTPUT_DIR', '/kaggle/working'))
CLUSTERING_KEYWORD = ('metric' if 'metric' in inspect.signature(_AgglomerativeClustering).parameters else 'affinity')
def AgglomerativeClustering(**kwargs):
    # Older sklearn renamed affinity to metric; the precomputed distances are identical.
    if CLUSTERING_KEYWORD == 'affinity':
        kwargs['affinity'] = kwargs.pop('metric')
    return _AgglomerativeClustering(**kwargs)
EXPECTED_VERSIONS = {'numpy': '2.5.4', 'scipy': '1.18.1', 'scikit-learn': '1.9.1', 'pandas': '3.0.6', 'joblib': '1.6.0', 'threadpoolctl': '3.7.0', 'pyarrow': '26.0.0'}
EXPECTED_PYTHON = '3.13.2'
ACTUAL_VERSIONS = {name: importlib.metadata.version(name) for name in EXPECTED_VERSIONS}
VERSION_MISMATCH = {name: {'local': value, 'runtime': ACTUAL_VERSIONS[name]}
                    for name, value in EXPECTED_VERSIONS.items() if ACTUAL_VERSIONS[name] != value}
if platform.python_version() != EXPECTED_PYTHON:
    VERSION_MISMATCH['python'] = {'local': EXPECTED_PYTHON, 'runtime': platform.python_version()}
print({'python': platform.python_version(), 'versions': ACTUAL_VERSIONS, 'version_mismatches': VERSION_MISMATCH})
ORIGINAL_CONFIG = {'classifier': 'logistic', 'C': 16.0, 'class_weight': None, 'lowercase': False, 'sublinear_tf': True, 'min_df': 2, 'char_ngram': [2, 5], 'word_ngram': [1, 2], 'representation': 'char', 'style_weight': 0.25}
CENTER_CONFIG = {'protocol': 'prompt-centering-dev-20261021', 'train_sha256': '40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707', 'split_sha256': 'c8c24921f617ad483b9e14cb23d0d64a0fbbde44c0214868445e8d0e30d6e1db', 'labels_sha256': 'd8c9dcad5979236fb0e5f5504dd2e53c676123bf75cbf06faa13bc1e86524b9d', 'baseline_oof_sha256': '40cf86da51cfaccc2ad6af396b62e1363c72ab27a871a3a1dca60c9be5feed75', 'baseline_contract_sha256': '4f8c614736e544884a3ccc362b015a4cd8356f375f93880e2b00dff08b466250', 'development_rows': 2159, 'development_groups': 120, 'folds': 5, 'max_fits': 15, 'max_seconds': 1800, 'max_rss_bytes': 3221225472, 'max_centered_csr_bytes': 734003200, 'threads': 2, 'seed': 20261021, 'char_max_features': 100000, 'word_max_features': 60000, 'word_weight': 0.5, 'cluster_distance_threshold': 0.65, 'cluster_max_features': 60000, 'C': 16.0, 'max_iter': 3000, 'models': [{'name': 'plain', 'strength': 0.0}, {'name': 'partial', 'strength': 0.5}, {'name': 'full', 'strength': 1.0}], 'blend_weights': [0.25, 0.5], 'bootstrap_seed': 20261021, 'bootstrap_repeats': 2000, 'adoption_min_delta': 0.005, 'adoption_min_positive_folds': 4, 'stress_min_delta': -0.005}
LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
EXPECTED_TRAIN_SHA256 = '40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707'
EXPECTED_TRAIN_ROWS = 2699
EXPECTED_TRAIN_GROUPS = 150
PROVENANCE = {'candidate': 'blend_partial_0.25', 'summary_sha256': 'b788bda9970a3a329d4111c75600eb8edd035caa6914df7dcd5e8353262675a5', 'contract_file_sha256': '9b0899a59dc21a0baed7e90fe5b1bd1a1f1afd791e07ad4ebd81644f83417e2b', 'source_sha256': {'scripts/model_cpu.py': 'fcfe8df6f52a077cdb5c8e0cddc7b917ef3354e2280854bb075059f0749cbcdc', 'scripts/model_cpu_round2.py': '5c1b2d1da22082d92137d25bf6a399428b9c0b98b944ad96fd3903a24ee509d1', 'scripts/compare_team_cpu.py': 'bac5a5f1a282a0a5d0a04cd5aaa92a5bced0cd17b71bb3e511ae0084baf8637e', 'scripts/compare_prompt_centering.py': 'ab41aebb1db7a3ad3501c9a7a45bcd47e190aa3cfd33277d5a0c61bc1fe7ec92', 'config/team_dev_20261021.json': 'c9f541de654da331f927bf951674ed6675859c2220d6d038ce9c40541f63c688', 'config/prompt_centering_20261021.json': '3e5e5e597e596482e38148bba89503bbef71a561fe4d590675926fa13ebcbcd3', 'config/labels.json': 'd8c9dcad5979236fb0e5f5504dd2e53c676123bf75cbf06faa13bc1e86524b9d'}, 'generator_sha256': '15226b52edda5aca79873d5829ab3a3acbce8893df5a1f5b3518931ff1150c4e', 'training_hash': '40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707', 'selection_scope': 'adaptive 120-prompt development; no confirmed improvement over previous best'}


In [ ]:
MODEL_SOURCES = {'model_cpu.py': '"""Reusable, offline TF-IDF classification and strict submission checks."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import FeatureUnion, Pipeline\nfrom sklearn.svm import LinearSVC\n\n\ndef read_csv(path):\n    with Path(path).open(encoding="utf-8-sig", newline="") as handle:\n        return list(csv.DictReader(handle))\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1 << 20), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef config_id(config):\n    return hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:12]\n\n\ndef feature_config(config):\n    return {key: value for key, value in config.items()\n            if key not in {"classifier", "C", "class_weight"}}\n\n\ndef make_features(config):\n    common = dict(lowercase=config.get("lowercase", False),\n                  sublinear_tf=config.get("sublinear_tf", True),\n                  min_df=config.get("min_df", 2), dtype=np.float32, norm="l2")\n    representations = []\n    if config["representation"] in {"char", "combined"}:\n        representations.append(("char", TfidfVectorizer(\n            analyzer="char", ngram_range=tuple(config.get("char_ngram", [3, 5])),\n            max_features=100000, **common)))\n    if config["representation"] in {"word", "combined"}:\n        representations.append(("word", TfidfVectorizer(\n            analyzer="word", ngram_range=tuple(config.get("word_ngram", [1, 2])),\n            token_pattern=r"(?u)\\b\\w+\\b", max_features=50000, **common)))\n    if not representations:\n        raise ValueError("Unknown representation")\n    return FeatureUnion(representations, n_jobs=1)\n\n\ndef make_classifier(config, seed=20261018):\n    common = dict(C=config.get("C", 1.0), class_weight=config.get("class_weight"),\n                  random_state=seed, tol=1e-4)\n    if config.get("classifier", "svm") == "svm":\n        return LinearSVC(dual=True, max_iter=10000, **common)\n    if config["classifier"] == "logistic":\n        return LogisticRegression(solver="lbfgs", max_iter=1000, **common)\n    raise ValueError("Unknown classifier")\n\n\ndef make_pipeline(config, seed=20261018):\n    return Pipeline([("features", make_features(config)),\n                     ("classifier", make_classifier(config, seed))])\n\n\ndef checked_submission(test_rows, predictions, labels, output):\n    """Preserve every test ID and actual row count; never infer hidden size."""\n    import pyarrow as pa\n    import pyarrow.parquet as pq\n    ids = [row["id"] for row in test_rows]\n    predictions = [str(value) for value in predictions]\n    if not ids or any(not str(value).strip() for value in ids) or len(set(ids)) != len(ids):\n        raise ValueError("Test IDs must be nonblank and unique")\n    if len(predictions) != len(ids) or any(value not in labels for value in predictions):\n        raise ValueError("Predictions must cover every test row with a reference label")\n    output = Path(output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    pq.write_table(pa.table({"id": ids, "model": predictions}), output)\n    result = pq.read_table(output).to_pydict()\n    if result["id"] != ids or result["model"] != predictions:\n        raise RuntimeError("Parquet round-trip validation failed")\n    return {"rows": len(ids), "unique_ids": len(set(ids)), "labels_valid": True,\n            "sha256": file_hash(output)}\n', 'model_cpu_round2.py': '"""Second CPU round: interpretable style features and independent score fusion."""\nfrom __future__ import annotations\n\nimport re\nimport numpy as np\nfrom sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin\nfrom sklearn.pipeline import FeatureUnion, Pipeline\nfrom sklearn.preprocessing import StandardScaler, Normalizer\n\nimport model_cpu as original\n\n\nclass StylometryFeatures(TransformerMixin, BaseEstimator):\n    """Text-only counts and rates. No IDs, prompts, labels or external resources."""\n    def fit(self, texts, y=None):\n        return self\n\n    def transform(self, texts):\n        rows = []\n        for text in texts:\n            words = re.findall(r"\\w+", text, flags=re.UNICODE)\n            lengths = np.array([len(word) for word in words], dtype=float)\n            n, count = max(len(text), 1), max(len(words), 1)\n            sentence_count = max(len(re.findall(r"[.!?]+", text)), 1)\n            values = [np.log1p(len(text)), np.log1p(len(words)), np.log1p(text.count("\\n")),\n                      float(lengths.mean()) if len(lengths) else 0.,\n                      float(lengths.std()) if len(lengths) else 0.,\n                      len(set(word.lower() for word in words)) / count,\n                      count / sentence_count, sum(char.isupper() for char in text) / n,\n                      sum(char.isdigit() for char in text) / n,\n                      sum(char.isspace() for char in text) / n,\n                      sum(ord(char) > 127 for char in text) / n]\n            values.extend(text.count(char) / n for char in\n                          ("\\n", ",", ";", ":", "!", "?", "(", "[", "-", "—", "\'", \'"\', "`", "*", "/"))\n            rows.append(values)\n        return np.asarray(rows, dtype=np.float32)\n\n\ndef make_features(config):\n    features = original.make_features(config)\n    if config.get("char_analyzer") == "char_wb":\n        features.set_params(char__analyzer="char_wb")\n    if config.get("style_weight", 0):\n        style = Pipeline([("counts", StylometryFeatures()), ("scale", StandardScaler()),\n                          ("normalize", Normalizer())])\n        features = FeatureUnion([("text", features), ("style", style)], n_jobs=1,\n                                transformer_weights={"text": 1., "style": config["style_weight"]})\n    return features\n\n\ndef normalize_scores(scores):\n    return scores / np.maximum(np.linalg.norm(scores, axis=1, keepdims=True), 1e-12)\n\n\nclass ScoreFusion(ClassifierMixin, BaseEstimator):\n    def __init__(self, char_config, word_config, char_weight=.5):\n        self.char_config = char_config\n        self.word_config = word_config\n        self.char_weight = char_weight\n\n    def fit(self, texts, y):\n        self.char_model_ = make_pipeline(self.char_config)\n        self.word_model_ = make_pipeline(self.word_config)\n        self.char_model_.fit(texts, y)\n        self.word_model_.fit(texts, y)\n        self.classes_ = self.char_model_.classes_\n        if not np.array_equal(self.classes_, self.word_model_.classes_):\n            raise ValueError("Fusion component classes differ")\n        return self\n\n    def decision_function(self, texts):\n        return (self.char_weight * normalize_scores(self.char_model_.decision_function(texts))\n                + (1 - self.char_weight) * normalize_scores(self.word_model_.decision_function(texts)))\n\n    def predict(self, texts):\n        return self.classes_[self.decision_function(texts).argmax(axis=1)]\n\n\ndef make_pipeline(config, seed=20261018):\n    if config.get("classifier") == "score_fusion":\n        return ScoreFusion(config["char_config"], config["word_config"], config["char_weight"])\n    return Pipeline([("features", make_features(config)),\n                     ("classifier", original.make_classifier(config, seed))])\n'}
original_cpu = types.ModuleType('model_cpu')
exec(compile(MODEL_SOURCES['model_cpu.py'], '<frozen model_cpu.py>', 'exec'), original_cpu.__dict__)
_prior_model_cpu = sys.modules.get('model_cpu')
sys.modules['model_cpu'] = original_cpu
try:
    style_cpu = types.ModuleType('model_cpu_round2')
    exec(compile(MODEL_SOURCES['model_cpu_round2.py'], '<frozen model_cpu_round2.py>', 'exec'), style_cpu.__dict__)
finally:
    if _prior_model_cpu is None:
        del sys.modules['model_cpu']
    else:
        sys.modules['model_cpu'] = _prior_model_cpu
make_pipeline = style_cpu.make_pipeline
read_csv = original_cpu.read_csv
file_hash = original_cpu.file_hash
checked_submission = original_cpu.checked_submission


In [ ]:
def soft_center(matrix, groups, strength, max_bytes):
    """Subtract each row's group mean; bound the union-support allocation first."""
    groups = np.asarray(groups)
    if groups.ndim != 1 or len(groups) != matrix.shape[0]:
        raise ValueError("One nonmissing group is required per row")
    if not np.isfinite(strength) or not 0 <= strength <= 1:
        raise ValueError("Centering strength must be in [0, 1]")
    if any(value is None or (isinstance(value, (float, np.floating)) and not np.isfinite(value))
           for value in groups):
        raise ValueError("Missing groups are not permitted")
    x = sparse.csr_matrix(matrix, dtype=np.float64, copy=True)
    if not len(groups):
        return x
    x = normalize(x, copy=False).tocsr()
    if strength == 0:
        return x
    _, inverse, counts = np.unique(groups, return_inverse=True, return_counts=True)
    # Singleton means are zero so single-item inference retains its input.
    weights = np.where(counts[inverse] > 1, 1.0 / counts[inverse], 0.0)
    membership = sparse.csr_matrix((weights, (inverse, np.arange(len(groups)))),
                                  shape=(len(counts), len(groups)))
    membership.eliminate_zeros()
    means = (membership @ x).tocsr()
    # Upper bound for union of input and repeated means, using 64-bit indices.
    estimated_nnz = x.nnz + int(np.dot(np.diff(means.indptr), counts))
    if estimated_nnz * 16 + (len(groups) + 1) * 8 > max_bytes:
        raise MemoryError("Centered CSR admission budget exceeded")
    centered = x - strength * means[inverse]
    centered.eliminate_zeros()
    return normalize(centered, copy=False).tocsr()

def cluster_batch(texts, ids, threshold, max_features):
    """Transductive text-only clustering, without prompt IDs or class labels."""
    ids, texts = np.asarray(ids), np.asarray(texts)
    if len(ids) != len(texts) or len(set(ids)) != len(ids):
        raise ValueError("Unique row IDs must align with texts")
    if not 0 < threshold < 1:
        raise ValueError("Cosine threshold must be in (0, 1)")
    order = np.argsort(ids, kind="stable")
    labels = np.arange(len(texts), dtype=np.int64)
    if len(texts) < 2:
        return labels
    try:
        x = TfidfVectorizer(ngram_range=(1, 2), min_df=1, stop_words="english",
             sublinear_tf=True, max_features=max_features, dtype=np.float64).fit_transform(texts[order])
    except ValueError as error:
        if "empty vocabulary" in str(error):
            return labels
        raise
    nonzero = np.flatnonzero(np.asarray(x.getnnz(axis=1)).ravel() > 0)
    if len(nonzero) < 2:
        return labels
    similarities = (x[nonzero] @ x[nonzero].T).toarray()
    distance = np.clip(1.0 - similarities, 0.0, 1.0)
    distance = (distance + distance.T) / 2
    np.fill_diagonal(distance, 0.0)
    clusters = AgglomerativeClustering(n_clusters=None, metric="precomputed",
               linkage="average", distance_threshold=threshold).fit_predict(distance)
    # Offset assigned labels to avoid collision with zero-vector singleton IDs.
    labels[order[nonzero]] = clusters + len(texts)
    return labels

def feature_pair(train_texts, valid_texts, config):
    train_blocks, valid_blocks = [], []
    for analyzer, ngram, weight, limit in (
            ("char_wb", (2, 5), 1.0, config["char_max_features"]),
            ("word", (1, 2), config["word_weight"], config["word_max_features"])):
        opts = dict(analyzer=analyzer, ngram_range=ngram, lowercase=True, min_df=2,
                    sublinear_tf=True, max_features=limit, dtype=np.float64)
        if analyzer == "word":
            opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
        vectorizer = TfidfVectorizer(**opts)
        train_blocks.append(vectorizer.fit_transform(train_texts) * weight)
        valid_blocks.append(vectorizer.transform(valid_texts) * weight)
    return tuple(normalize(sparse.hstack(blocks, format="csr"))
                 for blocks in (train_blocks, valid_blocks))
def validate_probabilities(probabilities, n_rows, n_labels):
    if probabilities.shape != (n_rows, n_labels) or probabilities.dtype.kind != "f":
        raise ValueError("Invalid probability schema")
    tolerance = max(1e-8, min(1e-6, np.finfo(probabilities.dtype).eps * n_labels))
    if (not np.isfinite(probabilities).all() or np.any(probabilities < 0)
            or np.any(probabilities > 1) or not np.allclose(probabilities.sum(axis=1), 1, atol=tolerance, rtol=0)):
        raise ValueError("Invalid or uncovered probability rows")

def canonical_probabilities(probabilities, n_rows, n_labels):
    """Accept ordinary float32 roundoff, then persist normalized float64 values."""
    validate_probabilities(probabilities, n_rows, n_labels)
    probabilities = probabilities.astype(np.float64, copy=True)
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    validate_probabilities(probabilities, n_rows, n_labels)
    return probabilities


In [ ]:

def atomic_json(path, value):
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        json.dump(value, handle, indent=2)
        handle.write('\n')
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def validate_inputs():
    train_path, test_path = INPUT / 'train.csv', INPUT / 'test.csv'
    if file_hash(train_path) != EXPECTED_TRAIN_SHA256:
        raise ValueError('Training hash differs from the frozen official snapshot')
    train, test = read_csv(train_path), read_csv(test_path)
    if not train or not test:
        raise ValueError('Train and test must both contain rows')
    # Conservative allowance for several dense float64 distance/similarity copies.
    # Refuse oversized batches rather than silently changing the frozen clustering.
    if 64 * len(test) ** 2 > 1073741824:
        raise MemoryError('Test clustering admission budget exceeded (64*n*n > 1 GiB)')
    if len(train) != EXPECTED_TRAIN_ROWS:
        raise ValueError('Training row count differs from the audited snapshot')
    if any(set(row) != {'id', 'prompt_id', 'text', 'model'} for row in train):
        raise ValueError('Training schema differs from the audited snapshot')
    if any(set(row) != {'id', 'text'} for row in test):
        raise ValueError('Test schema must be exactly id and text')
    for name, rows in (('train', train), ('test', test)):
        if any(not isinstance(row['id'], str) or not row['id'].strip()
               or not isinstance(row['text'], str) or not row['text'].strip() for row in rows):
            raise ValueError(name + ' IDs and texts must be nonblank strings')
        if len({row['id'] for row in rows}) != len(rows):
            raise ValueError(name + ' IDs must be unique')
    if len(LABELS) != 18 or len(set(LABELS)) != 18 or set(row['model'] for row in train) != set(LABELS):
        raise ValueError('Training classes differ from the frozen 18-label reference')
    if any(not isinstance(row['prompt_id'], str) or not row['prompt_id'].strip() for row in train):
        raise ValueError('Training prompts must be nonblank')
    if len({row['prompt_id'] for row in train}) != EXPECTED_TRAIN_GROUPS:
        raise ValueError('Training prompt count differs from audited snapshot')
    return train, test, {'train_sha256': file_hash(train_path), 'test_sha256': file_hash(test_path)}


def assert_classes(model):
    if not np.array_equal(model.classes_, np.arange(len(LABELS))):
        raise ValueError('Classifier class order differs from canonical label order')


def run_candidate():
    train, test, input_hashes = validate_inputs()
    OUTPUT.mkdir(parents=True, exist_ok=True)
    artifacts = ('submission.parquet', 'probabilities.npz', 'runtime.json')
    if any((OUTPUT / name).exists() for name in artifacts):
        raise FileExistsError('Candidate output already exists; refusing overwrite or silent rerun')
    claim_path = OUTPUT / 'candidate_attempt.json'
    claim = {'state': 'running', 'candidate': 'blend_partial_0.25', 'max_classifier_fits': 2,
             'attempted_fits': 0, 'completed_fits': 0, 'formal_submission': False,
             'input_hashes': input_hashes, 'provenance': PROVENANCE}
    # Exclusive creation is the persistent admission guard; never delete/reset a claim.
    with claim_path.open('x', encoding='utf-8') as handle:
        json.dump(claim, handle, indent=2)
        handle.flush()
        os.fsync(handle.fileno())
    started, fit_records = time.monotonic(), []
    try:
        texts = np.asarray([row['text'] for row in train])
        test_texts = np.asarray([row['text'] for row in test])
        ids = np.asarray([row['id'] for row in test])
        truth = np.asarray([LABELS.index(row['model']) for row in train])
        groups = np.asarray([row['prompt_id'] for row in train])
        def guarded_fit(model, x, name):
            if claim['attempted_fits'] >= 2:
                raise RuntimeError('Exactly two classifier fits are authorized')
            claim['attempted_fits'] += 1
            claim['current_model'] = name
            atomic_json(claim_path, claim)
            fit_started = time.monotonic()
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always', ConvergenceWarning)
                model.fit(x, truth)
            claim['completed_fits'] += 1
            atomic_json(claim_path, claim)
            convergence = sum(issubclass(w.category, ConvergenceWarning) for w in caught)
            fit_records.append({'model': name, 'seconds': time.monotonic() - fit_started,
                                'convergence_warnings': convergence})
            if convergence:
                raise RuntimeError('Convergence warning; do not retune or retry implicitly')
            assert_classes(model)
            return model
        with threadpool_limits(limits=2):
            original_model = guarded_fit(make_pipeline(ORIGINAL_CONFIG, seed=20261018), texts, 'original_style')
            original_proba = canonical_probabilities(original_model.predict_proba(test_texts), len(test), len(LABELS))
            del original_model
            gc.collect()
            train_x, test_x = feature_pair(texts, test_texts, CENTER_CONFIG)
            train_x = soft_center(train_x, groups, 0.5, CENTER_CONFIG['max_centered_csr_bytes'])
            partial_model = guarded_fit(LogisticRegression(C=16.0, class_weight='balanced',
                            max_iter=3000, random_state=20261021), train_x, 'partial')
            del train_x
            gc.collect()
            test_groups = cluster_batch(test_texts, ids, 0.65, CENTER_CONFIG['cluster_max_features'])
            test_x = soft_center(test_x, test_groups, 0.5, CENTER_CONFIG['max_centered_csr_bytes'])
            partial_proba = canonical_probabilities(partial_model.predict_proba(test_x), len(test), len(LABELS))
            blend = canonical_probabilities(0.75 * original_proba + 0.25 * partial_proba, len(test), len(LABELS))
            del partial_model, test_x
        predictions = [LABELS[i] for i in blend.argmax(axis=1)]
        checks = checked_submission(test, predictions, LABELS, OUTPUT / 'submission.parquet')
        np.savez_compressed(OUTPUT / 'probabilities.npz', ids=ids, labels=np.asarray(LABELS),
                            original_style=original_proba, partial=partial_proba, blend=blend)
        runtime = {'candidate': 'blend_partial_0.25', 'weights': {'original_style': 0.75, 'partial': 0.25},
            'original_config': ORIGINAL_CONFIG, 'centering_config': CENTER_CONFIG,
            'train_rows': len(train), 'train_prompts': len(set(groups)), 'test_rows': len(test),
            'public_test_example_rows': 3, 'is_three_row_interface_example': len(test) == 3,
            'test_size_is_not_assumed_hidden_size': True,
            'test_cluster_sizes': sorted(np.unique(test_groups, return_counts=True)[1].tolist()),
            'elapsed_seconds': time.monotonic() - started, 'fit_records': fit_records,
            'full_training_classifier_fits': claim['completed_fits'], 'attempted_fits': claim['attempted_fits'],
            'input_hashes': input_hashes, 'provenance': PROVENANCE,
            'python': platform.python_version(), 'versions': ACTUAL_VERSIONS,
            'version_mismatches': VERSION_MISMATCH, 'exact_environment_match': not VERSION_MISMATCH,
            'clustering_keyword': CLUSTERING_KEYWORD, 'output_validation': checks,
            'clustering_admission_bytes': 64 * len(test) ** 2,
            'probabilities_sha256': file_hash(OUTPUT / 'probabilities.npz'),
            'final_holdout_evaluations': 0, 'formal_submission': False,
            'interpretation': 'Full-data fit and interface prediction; no new validation or leaderboard score'}
        atomic_json(OUTPUT / 'runtime.json', runtime)
        claim.update(state='completed', elapsed_seconds=runtime['elapsed_seconds'],
                     runtime_sha256=file_hash(OUTPUT / 'runtime.json'), output_validation=checks)
        atomic_json(claim_path, claim)
        print(json.dumps({key: runtime[key] for key in ('candidate', 'train_rows', 'test_rows',
              'full_training_classifier_fits', 'elapsed_seconds', 'version_mismatches',
              'output_validation', 'formal_submission')}, indent=2))
        return runtime
    except BaseException as error:
        claim.update(state='failed', error=type(error).__name__ + ': ' + str(error),
                     elapsed_seconds=time.monotonic() - started, fit_records=fit_records)
        atomic_json(claim_path, claim)
        raise


In [ ]:
runtime = run_candidate()
